# OpenBookMap — YOLOv8n spine detector

Fine-tune a single-class `spine` detector on a public Roboflow book-spine dataset, export to ONNX, and download the weights to drop into `models/spine-yolov8n.onnx` in the repo.

**Runtime:** Colab (T4 GPU is enough). ~20–40 min wall-clock end to end.

**Output:** `spine-yolov8n.onnx` (~12 MB) with NMS baked in — output shape `[1, N, 6]` = `(x1, y1, x2, y2, conf, cls)`.

Commit the resulting file to the repo at `models/spine-yolov8n.onnx`.

## 1. Install deps

`ultralytics>=8.3.0` bakes NMS into ONNX when `nms=True`. Older versions silently ignore the flag — pin the version.

In [5]:
!pip install -q "ultralytics>=8.3.0" roboflow onnx onnxsim onnxruntime

^C


## 2. Pull a spine dataset from Roboflow Universe

Get a free API key at https://app.roboflow.com/settings/api and paste below.

**Default dataset:** `capjamesg/book-spines`. Roboflow slugs and versions drift — if the download below 404s or the images look low quality, browse https://universe.roboflow.com/ and search `book spine` / `bookshelf`, then copy the YOLOv8 download snippet from your chosen project into the **fallback** cell.

You want: YOLOv8 format, single `spine` class (or remap multi-class datasets to one class in the training step).

In [ ]:
import os
from roboflow import Roboflow

os.environ["ROBOFLOW_API_KEY"] = "rf_inM2uyMzzGXYta086UYGVCWV9p42"

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("capjamesg").project("book-spines")
dataset = project.version(1).download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print("data.yaml:", DATA_YAML)
!cat {DATA_YAML}

ModuleNotFoundError: No module named 'roboflow'

### Fallback: paste a different Roboflow snippet here if the default is gone

Uncomment and replace `<workspace>` / `<project>` / `<version>` from the Roboflow Universe page's "Download" → "YOLOv8" snippet.

In [ ]:
# rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
# project = rf.workspace("<workspace>").project("<project>")
# dataset = project.version(<version>).download("yolov8")
# DATA_YAML = f"{dataset.location}/data.yaml"
# print(DATA_YAML); !cat {DATA_YAML}

## 3. Fine-tune YOLOv8n

Single class, 640 imgsz, cosine LR, early stopping. 60 epochs is usually plenty for one class; `patience=10` stops early if mAP plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=DATA_YAML,
    imgsz=640,
    epochs=60,
    batch=16,
    patience=10,
    cos_lr=True,
    project="runs",
    name="spine",
    exist_ok=True,
)

## 4. Validate

Target: **mAP50 ≥ 0.85**. Below that, the detector will miss spines at runtime — try more epochs, a larger model (`yolov8s.pt`), or a different dataset.

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 5. Export to ONNX with NMS baked in

`nms=True` gives output shape `[1, N, 6]` — rows of `(x1, y1, x2, y2, conf, cls)` in letterboxed 640×640 coords. The browser code in `upload.html` assumes this shape.

In [ ]:
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    nms=True,
    simplify=True,
    opset=12,
    dynamic=False,
)
print("exported to:", onnx_path)
!ls -lh {onnx_path}

## 6. Quick sanity check — run the ONNX in Python

Confirms the graph works and the output shape matches what `detectSpines()` in `upload.html` expects.

In [ ]:
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
out = sess.run(None, {sess.get_inputs()[0].name: dummy})
print("output shapes:", [o.shape for o in out])
print("expected shape like [1, N, 6] (N = detection count after NMS)")

## 7. Download

Save to your machine, then commit as `models/spine-yolov8n.onnx` in the OpenBookMap repo.

In [ ]:
import shutil
shutil.copy(onnx_path, "spine-yolov8n.onnx")

try:
    from google.colab import files
    files.download("spine-yolov8n.onnx")
except ImportError:
    print("Not on Colab — file saved as ./spine-yolov8n.onnx")